# Quickstart

Este notebook muestra el uso mínimo de `pryngles` usando la interfaz **`System`** para construir un sistema estrella–planeta–anillo y calcular una curva de luz simple.

## Instalación (solo Colab)

Si estás en Google Colab, ejecuta una de estas opciones:

In [ ]:
# %pip install -Uq pryngles
# %pip install -Uq git+https://github.com/seap-udea/pryngles.git

## Ejemplo mínimo

In [ ]:
import numpy as np
import pryngles as pr
from pryngles import Consts

# Construir un sistema
system = pr.System()

star = system.add(kind="Star", radius=Consts.rsun/system.ul, limb_coeffs=[0.65])
planet = system.add(
    kind="Planet",
    primary=star,
    a=0.2,
    e=0.0,
    radius=Consts.rsaturn/system.ul,
    nspangles=1000,
)
ring = system.add(
    kind="Ring",
    primary=planet,
    fi=1.5,
    fe=2.5,
    i=30 * Consts.deg,
    nspangles=1000,
)

system.initialize_simulation()

# Preparar malla/superficies (spangling)
system.spangle_system()

# Curva de luz básica (tránsito)
times = np.linspace(-0.5, 0.5, 51)
out = system.compute_lightcurve(times=times, effects=["transit"])

out["transit"].head()

Si quieres el flujo de trabajo completo (tránsito + emisión térmica + polarización) revisa el tutorial **System Interface**.

## Realistic scattering y polarización

Este ejemplo calcula el flujo reflejado y el grado de polarización con un modelo de dispersión más realista. (Si estás en Colab, añade primero `%matplotlib inline`.)

In [ ]:
import matplotlib.pyplot as plt

# Observador (lambda_ecl, beta_ecl) en radianes
observer = (-90 * Consts.deg, 60 * Consts.deg)

# Un periodo orbital (en unidades internas de rebound)
P = system.sim.particles[planet.name].P
nt = 60
times = np.linspace(0.0, P, nt)

out = system.compute_lightcurve(
    times=times,
    effects=["reflection", "polarization"],
    bodies=[planet.name, ring.name],
    observer=observer,
)

ref = out["reflection"]
pol = out["polarization"]

flux_planet = ref[(planet.name, "reflection")].to_numpy(dtype=float)
flux_ring = ref[(ring.name, "reflection")].to_numpy(dtype=float)
flux_total = flux_planet + flux_ring

P_planet = pol[(planet.name, "polarization")].to_numpy(dtype=float)
P_ring = pol[(ring.name, "polarization")].to_numpy(dtype=float)

phase_deg = np.linspace(0.0, 360.0, nt)

plt.close('all')
fig, axs = plt.subplots(2, 1, figsize=(7, 6), sharex=True)

axs[0].plot(phase_deg, Consts.ppm * flux_planet, label="Planet")
axs[0].plot(phase_deg, Consts.ppm * flux_ring, label="Ring")
axs[0].plot(phase_deg, Consts.ppm * flux_total, label="Planet+Ring")
axs[0].set_ylabel("Flux anomaly [ppm]")
axs[0].legend()
axs[0].grid(True, alpha=0.3)

axs[1].plot(phase_deg, P_planet, label="Planet")
axs[1].plot(phase_deg, P_ring, label="Ring")
axs[1].set_ylabel("Degree of polarization [-]")
axs[1].set_xlabel("Orbital phase [deg]")
axs[1].legend()
axs[1].grid(True, alpha=0.3)

fig.tight_layout()
plt.show()